# 05 ส่งต่อโมเดลสู่แอป ทดสอบ และติดตาม
ก่อนเริ่ม: มี pipeline และ schema ที่กลุ่มเลือก/ประเมินแล้วจาก 04 ยังไม่มีโค้ดโหลด artifact สำเร็จรูป เพราะกลุ่มต้องกำหนดวิธีและรุ่นไฟล์เอง


## เจ้าของงานและการส่งต่อสำหรับทีม 5 คน
เจ้าของช่วงนี้ M5: รับ final artifact/schema/metadata จาก M2 ให้ M4 ตรวจ prediction parity และ input tests ให้ M1 ตรวจหน่วย/ความหมาย แล้ว deploy บน Community Cloud และเก็บ monitoring ร่วม M1

M1–M5 เป็นรหัสบทบาทเสนอ ให้กลุ่มใส่ชื่อจริงและตกลงผู้ตรวจ ไม่ใช่หลักฐานว่างานทำเสร็จแล้ว ทุกคนบันทึก contribution/prompts และเขียนรายงานส่วนของตน


## เตรียมงานใน Colab
บันทึกสำเนา notebook ใน Drive ก่อน รัน cell นี้เพื่อเตรียม repository ใน runtime เมื่อมีโฟลเดอร์เดิมจะใช้ต่อโดยไม่เขียนทับ สำรองไฟล์ใน Files แยกจาก notebook ก่อนปิด runtime

ใช้ CPU runtime ได้ ไม่ต้องติดตั้งโปรแกรมในเครื่อง


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    if not Path('/content').is_dir():
        raise RuntimeError('เปิด notebook นี้บน Google Colab และเชื่อม runtime ก่อน')
    ROOT = Path('/content/Bicycle_Rental_Prediction')
    if ROOT.exists():
        raise FileExistsError('พบโฟลเดอร์เดิมที่ยังอ่าน config ไม่ได้ ตรวจและสำรองงานก่อนแก้')
    subprocess.run(['git', 'clone',
                    'https://github.com/Supot77/Bicycle_Rental_Prediction.git',
                    str(ROOT)], check=True)

# Keep Colab's existing scientific stack; install only packages that are missing.
packages = {'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('พร้อมทำงานบน Colab: ใช้โฟลเดอร์ repository เดิมถ้ามี โดยไม่ pull/reset ทับงาน')
print('ค่า null ใน config เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


## โหลด artifact ใหม่
เขียนโค้ดโหลด preprocessing+model และลอง input ที่เตรียมจาก schema จริง บันทึกผลอ้างอิงจาก notebook ที่ทำซ้ำได้ก่อนต่อ app.py

ถ้าแยกหลายไฟล์ต้องตรวจว่าเป็นรุ่นเดียวกัน ไม่โหลด pickle/joblib จากแหล่งที่ไม่เชื่อถือ


In [ ]:
# เขียนการโหลด artifact ที่กลุ่มสร้าง และตัวอย่าง prediction ที่ตรวจแล้ว
# ยังไม่ใส่ชื่อโมเดลหรือผลสมมติในโครงเริ่มต้น


## ตรวจโครงแอปได้จาก Colab
ตัวอย่างนี้ตรวจว่า app.py เปิดได้โดยไม่มี exception โดยจำลองแอปภายใน notebook ยังไม่ใช่การตรวจ UI ทุกกรณีหรือเปิดแอปสาธารณะ สมาชิกต้องเพิ่มกรณี input และ prediction หลังเขียนส่วนจริง

ถ้า Colab แจ้งให้ restart หลังติดตั้ง ให้สำรองงานแล้วทำตาม จากนั้นรัน cell เตรียมงานใหม่


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install',
                '-r', str(ROOT / 'requirements-colab.txt')], check=True)
from streamlit.testing.v1 import AppTest
app_check = AppTest.from_file(str(ROOT / 'app.py'), default_timeout=30).run()
assert not app_check.exception, app_check.exception
print('app.py รันผ่าน AppTest; ยังต้องตรวจ input/prediction และ UI จริงตามแผน')


## ลงมือใน app.py
เพิ่มช่องกรอกตามหน่วยจริง mapping ของ feature validation และปุ่มทำนาย นำสถานะรอข้อมูลออกเมื่อมีระบบจริงครบแล้ว แสดงหน่วยและข้อจำกัด ใช้ pipeline เดียวกับ notebook

อธิบายกฎผลติดลบ/ปัดเศษถ้ามี และใช้กฎเดียวกันในการประเมิน ห้ามฝึกใหม่ทุกครั้งที่กดปุ่ม


## ออกแบบและรันทดสอบ
ให้สมาชิกเขียนกรณี normal/boundary/invalid พร้อม expected ก่อนรัน บันทึก actual และหลักฐานใน evidence/test_log.csv

เทียบค่าดิบ notebook/dashboard ด้วย tolerance ที่ตกลง ตรวจ unknown categories input ไม่ครบ artifact ไม่พบ และเปิดแอปจาก session ใหม่ การมีกรณีทดสอบในเอกสารยังไม่เท่ากับผ่านการทดสอบ


In [ ]:
# เขียนการตรวจเทียบผลและ validation ที่เกี่ยวข้องกับ implementation จริงของกลุ่ม


## Deploy และ monitoring
ทำตาม docs/DEPLOYMENT_TH ตรวจลิงก์จริงและบันทึกรุ่น ดู logs หรือบันทึกที่ไม่อ่อนไหว กำหนด review trigger จากบริบท

เขียนสิ่งที่พบจริงใน monitoring.csv หากไม่มี actual outcome ของ prediction ใหม่ ให้รายงานพฤติกรรมระบบและ input ได้ แต่ยังวัดความแม่นจริงไม่ได้


## สิ่งส่งต่อและจุดตรวจ
แอปออนไลน์ test log ผลเทียบ notebook monitoring record README ฉบับทำตามได้ และหลักฐานสมาชิก ให้เพื่อนที่ไม่ได้เขียนส่วนนี้รันจากต้นใน Colab และเปิด UI จริงบน Community Cloud ก่อนซ้อมสาธิต


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
